# Long term adaptation planning - Physical damages

The Damages module in RA2CE estimates the physical (direct) damage to infrastructure caused by natural hazards such as floods, cyclones, or earthquakes. This provides a critical step in risk assessment: quantifying how exposed assets will be affected when hazard intensities exceed certain thresholds.

## 🚑 What is the direct cost of physical damages from hazard on infrastructures ?


Two main types of questions can be addressed:

- Event-based damages – What is the expected physical damage to my infrastructure for a specific hazard event? <br>

- Risk-based damages – What is the *Expected Annual Damage (EAD)* to my infrastructure, based on hazard probability distributions? <br>




## Initialize and import RA2CE functions for the analysis

General imports

In [ ]:
#python imports
from pathlib import Path
import geopandas as gpd
import folium 
from shapely.geometry import Point,LineString, Polygon, box
import rasterio
import matplotlib.pyplot as plt
import osmnx 
import pandas as pd
from shapely.geometry import shape
import os
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np
from folium.raster_layers import ImageOverlay

RA2CE-specific imports from the RA2CE python package

In [ ]:
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.network_config_data import NetworkSection
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.network_config_data import HazardSection
from ra2ce.network.network_config_data.network_config_data import NetworkConfigData

from ra2ce.analysis.damages.damages import AnalysisSectionDamages
from ra2ce.analysis.analysis_config_data.enums.risk_calculation_mode_enum import RiskCalculationModeEnum
from ra2ce.analysis.analysis_config_data.enums.analysis_damages_enum import AnalysisDamagesEnum
from ra2ce.analysis.analysis_config_data.enums.event_type_enum import EventTypeEnum
from ra2ce.analysis.analysis_config_data.enums.damage_curve_enum import DamageCurveEnum
from ra2ce.analysis.analysis_config_data.analysis_config_data import AnalysisConfigData

from ra2ce.ra2ce_handler import Ra2ceHandler

Set your working directories according to the RA2CE folder structure

In [ ]:
root_dir = Path('C:/RA2CE_Launch/data') # Adapt this path to your local directory where you have the data stored.
static_path=root_dir.joinpath("static")
network_path=static_path.joinpath("network")
hazard_path=static_path.joinpath("hazard")
output = root_dir.joinpath("output")

print('Our working directory is :' + str(root_dir))

# Step 1 Data preparation

## Area of interest

In [ ]:
from IPython.display import HTML

#Determine the extent:
AOI = network_path.joinpath("polygon_miami_beach.shp")
AOI_gdf = gpd.read_file(AOI)
m = AOI_gdf.explore(width=700, height=500)

OD_polygon = shape(AOI_gdf.geometry.iloc[0])

HTML(f"""
<div style="display:flex; justify-content:center; width:100%;">
  <div style="width:700px; height:500px; overflow:hidden;">
    {m._repr_html_()}
  </div>
</div>
""")

## Vulnerability curve (hazard severity vs. damage fraction)

In the manual situation, two input files are expected:

1. A file that specifies the **shape of the vulnerability curve**:
   - x-axis = hazard intensity (e.g. water depth in cm)
   - y-axis = damage fraction (0–1, representing the % of total construction cost)

2. A file that specifies the **construction costs** per road type and number of lanes.

Both files should be placed in the folder `input_data/damage_functions/all_road_types/`.

The file `hazard_severity_damage_fraction.csv` looks like:
```text
depth;damage
cm;%
0.0;0.0
50;0.12
100;0.2
200;0.28
600;0.35
```

Where:
- `depth` = water depth in **cm**
- `damage` = damage fraction (0–1, relative to construction cost)

In [ ]:
input_data_path = root_dir.joinpath("input_data")

vuln_curves = pd.read_csv(
    input_data_path.joinpath("damage_functions", "all_road_types","hazard_severity_damage_fraction.csv"),
    delimiter=";",
    skiprows=[1]  # skip units row (cm;%)
)

# Ensure numeric values
vuln_curves["depth"] = pd.to_numeric(vuln_curves["depth"])
vuln_curves["damage"] = pd.to_numeric(vuln_curves["damage"])

# Sort values (important for interpolation)
vuln_curves = vuln_curves.sort_values("depth")

# Original data
depth = vuln_curves["depth"].values
damage = vuln_curves["damage"].values

# Create smooth depth range for interpolation
depth_interp = np.linspace(depth.min(), depth.max(), 200)

# Linear interpolation
damage_interp = np.interp(depth_interp, depth, damage)

# Plot
plt.figure(figsize=(7,5))

# Original points
plt.scatter(depth, damage, label="Original data", zorder=3)

# Interpolated curve
plt.plot(depth_interp, damage_interp, label="Interpolated curve")

plt.xlabel("Water Depth [cm]")
plt.ylabel("Damage fraction")
plt.title("Interpolated vulnerability curve")
plt.grid(True)
plt.legend()

plt.show()

### Maximum construction costs per road type and lanes

The file `max_damage_road_types.csv` looks like:
```text
Road_type \ lanes;1;2;3;4;5;6;7;8
unit;$/m;$/m;$/m;$/m;$/m;$/m;$/m;$/m
tertiary_link;110;120;130;140;150;130;140;150
tertiary;110;120;130;140;150;130;140;150
trunk;110;120;130;140;150;130;140;150
trunk_link;110;120;130;140;150;130;140;150
secondary_link;11;12;13;14;15;13;14;15
secondary;11;12;13;14;15;13;14;15
primary_link;11;12;13;14;15;13;14;15
primary;11;12;13;14;15;13;14;15
residential;1100;1200;1300;1400;1500;1300;1400;1500
motorway;1100;1200;1300;1400;1500;1300;1400;1500
motorway_link;510;520;530;540;550;530;540;550
```

----
## Step 2: Configure the road network and hazard

The network is downloaded from **OpenStreetMap (OSM)**, clipped to a region polygon (`polygon_miami_beach.shp`).  
We specify which **road types** should be included in the analysis.

In [ ]:
network_section = NetworkSection(
    source=SourceEnum.OSM_DOWNLOAD,
    polygon=AOI,
    road_types=[
        RoadTypeEnum.SECONDARY,
        RoadTypeEnum.SECONDARY_LINK,
        RoadTypeEnum.PRIMARY,
        RoadTypeEnum.PRIMARY_LINK,
        RoadTypeEnum.TRUNK,
        RoadTypeEnum.MOTORWAY,
        RoadTypeEnum.MOTORWAY_LINK,
        RoadTypeEnum.RESIDENTIAL
    ],
    save_gpkg=True,
)

hazard_section = HazardSection(
    hazard_map=[Path(file) for file in hazard_path.glob("*.tif")],
    aggregate_wl=AggregateWlEnum.MEAN,  #we take the mean water depth on road segments that intersect with the flood map
    hazard_crs="EPSG:4326",  # ensure hazard map is in EPSG:4326 projection
)

network_config_data = NetworkConfigData(
    root_path=root_dir,
    static_path=static_path,
    network=network_section,
    hazard=hazard_section
)



> **Note**: The damage analysis requires two attributes from the network file: highway (for the type of roads) and lanes (number of lanes). These attributes are present by default from OSM. Make sure these are also present if you are using your own shapefile.

----
## Step 3: Define the damage analysis - event based

Here, we specify that RA2CE should perform a **damage analysis** using **manual damage curves (MAN)** with the class [AnalysisSectionDamages](../api/ra2ce.analysis.analysis_config_data.html#ra2ce.analysis.analysis_config_data.analysis_config_data.AnalysisSectionDamages) and the attribute `damage_curve` set to [DamageCurveEnum.MAN](../api/ra2ce.analysis.analysis_config_data.enums.html#ra2ce.analysis.analysis_config_data.enums.damage_curve_enum.DamageCurveEnum.MAN)
For manual damage curves, it is important to also specify the input data in the folder path in the config `input_path`. This is the location where the custom manual curves will be defined and placed (see next step).

The event type can be set to EVENT if damages are to be calculated for the hazard maps only (example below), or to RETURN_PERIOD if the analysis should estimate risk over a specified return period (see tutorial ).

In [ ]:
damages_analysis = [AnalysisSectionDamages(
    name='damages_case_florida_event_based',
    analysis=AnalysisDamagesEnum.DAMAGES,
    event_type=EventTypeEnum.EVENT,
    damage_curve=DamageCurveEnum.MAN,  # use manual damage curve
    save_csv=True,
    save_gpkg=True
)]

analysis_config_data = AnalysisConfigData(
    analyses=damages_analysis,
    output_path=output,
    input_path=root_dir.joinpath("input_data"),
)

In [ ]:
# network_config_data.network.reuse_network_output = True
Ra2ceHandler.run_with_config_data(network_config_data, analysis_config_data)

----
## Output

The results of the manual damage analysis are provided in **two GeoPackage (GPKG) files**:

- `damages_reference_florida_event_based_link_based.gpkg`: damage estimates per **network link**
- `damages_reference_florida_event_based_segment.gpkg`: damage estimates per **100m segment**

Key attributes of interest (in currency):

- `dam_RP20_al` : estimated damage for the first flood map (manual method).
- `dam_RP20_al` : estimated damage for the second flood map (manual method).

You can open these files in GIS software (QGIS, ArcGIS) or load them in Python with GeoPandas:

In [ ]:
gpkg_path = static_path.joinpath('output_graph', "base_network_hazard.gpkg")

gdf = gpd.read_file(gpkg_path)

map = gdf.explore(column="RP20_ma", cmap="Blues", legend=True, width=700, height=500, tiles="CartoDB Positron")
map


In [ ]:
import geopandas as gpd
output_path = root_dir / "output" / 'damages'
link_based = gpd.read_file(output_path / "damages_case_florida_event_based_link_based.gpkg")
segment_based = gpd.read_file(output_path / "damages_case_florida_event_based_segmented.gpkg")

# Inspect the first rows
print(segment_based.columns)


In [ ]:
import branca.colormap as cm
gpkg_path = output.joinpath("damages", "damages_case_florida_event_based_segmented.gpkg")
value_column = "dam_RP20_al"


gdf = gpd.read_file(gpkg_path)

if value_column not in gdf.columns:
    raise KeyError(f"Column '{value_column}' not found. Available columns: {list(gdf.columns)}")

gdf_plot = gdf.copy()
if gdf_plot.crs is None:
    raise ValueError("Input GeoPackage has no CRS defined.")
if gdf_plot.crs.to_epsg() != 4326:
    gdf_plot = gdf_plot.to_crs(epsg=4326)

gdf_plot[value_column] = pd.to_numeric(gdf_plot[value_column], errors="coerce")
gdf_plot = gdf_plot.dropna(subset=[value_column])

vmin, vmax = 0, 50000
tick_values = list(range(0, 50001, 2000))

m_dam_ev4 = folium.Map(
    location=[gdf_plot.geometry.centroid.y.mean(), gdf_plot.geometry.centroid.x.mean()],
    zoom_start=12,
    tiles="CartoDB Positron",
    width=900,
    height=600,
)

colormap = cm.LinearColormap(
    colors=cm.linear.YlOrRd_09.colors,
    vmin=vmin,
    vmax=vmax,
    caption="Direct damage $",
).to_step(index=tick_values)

def style_function(feature):
    value = feature["properties"].get(value_column)
    if value is None or pd.isna(value):
        color = "#bdbdbd"
    else:
        value = max(vmin, min(vmax, float(value)))
        color = colormap(value)
    return {"color": color, "weight": 3, "opacity": 0.95}

folium.GeoJson(
    data=gdf_plot.to_json(),
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=[value_column], aliases=["cost $:"]),
).add_to(m_dam_ev4)

colormap.add_to(m_dam_ev4)
m_dam_ev4

In [ ]:
import branca.colormap as cm

gpkg_path = output.joinpath("damages", "damages_case_florida_event_based_link_based.gpkg")
value_column = "dam_RP20_al"

gdf = gpd.read_file(gpkg_path)

if value_column not in gdf.columns:
    raise KeyError(f"Column '{value_column}' not found. Available columns: {list(gdf.columns)}")

gdf_plot = gdf.copy()
if gdf_plot.crs is None:
    raise ValueError("Input GeoPackage has no CRS defined.")
if gdf_plot.crs.to_epsg() != 4326:
    gdf_plot = gdf_plot.to_crs(epsg=4326)

gdf_plot[value_column] = pd.to_numeric(gdf_plot[value_column], errors="coerce")
gdf_plot = gdf_plot.dropna(subset=[value_column])

vmin, vmax = 0, 200000
tick_values = list(range(0, 200001, 20000))

m_dam_ev4 = folium.Map(
    location=[gdf_plot.geometry.centroid.y.mean(), gdf_plot.geometry.centroid.x.mean()],
    zoom_start=12,
    tiles="CartoDB Positron",
    width=900,
    height=600,
)

colormap = cm.LinearColormap(
    colors=cm.linear.YlOrRd_09.colors,
    vmin=vmin,
    vmax=vmax,
    caption="Direct damage $",
).to_step(index=tick_values)

def style_function(feature):
    value = feature["properties"].get(value_column)
    if value is None or pd.isna(value):
        color = "#bdbdbd"
    else:
        value = max(vmin, min(vmax, float(value)))
        color = colormap(value)
    return {"color": color, "weight": 3, "opacity": 0.95}

folium.GeoJson(
    data=gdf_plot.to_json(),
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=[value_column], aliases=["cost $:"]),
).add_to(m_dam_ev4)

colormap.add_to(m_dam_ev4)
m_dam_ev4

You can open the results in GIS software to visualize which road segments are most affected by the hazard.

----
## Step 4: Define the damage analysis - risk based


The event type can be set to EVENT if damages are to be calculated for the hazard maps only (example below), or to RETURN_PERIOD if the analysis should estimate risk over a specified return period (see tutorial ).

In [ ]:
damages_analysis = [AnalysisSectionDamages(
    name='damages_case_florida_risk',
    analysis=AnalysisDamagesEnum.DAMAGES,
    event_type=EventTypeEnum.RETURN_PERIOD,
    damage_curve=DamageCurveEnum.MAN,  # use manual damage curve
    risk_calculation_mode=RiskCalculationModeEnum.DEFAULT,
    save_csv=True,
    save_gpkg=True
)]

analysis_config_data = AnalysisConfigData(
    analyses=damages_analysis,
    output_path=output,
    input_path=root_dir.joinpath("input_data"),
)

In [ ]:
Ra2ceHandler.run_with_config_data(network_config_data, analysis_config_data)

In [ ]:
import branca.colormap as cm

gpkg_path = output.joinpath("damages", "damages_case_florida_risk_segmented.gpkg")
value_column = "risk_al" ## This is the column with the risk values, change if you want to plot a different column

gdf = gpd.read_file(gpkg_path)

if value_column not in gdf.columns:
    raise KeyError(f"Column '{value_column}' not found. Available columns: {list(gdf.columns)}")

gdf_plot = gdf.copy()
if gdf_plot.crs is None:
    raise ValueError("Input GeoPackage has no CRS defined.")
if gdf_plot.crs.to_epsg() != 4326:
    gdf_plot = gdf_plot.to_crs(epsg=4326)

gdf_plot[value_column] = pd.to_numeric(gdf_plot[value_column], errors="coerce")
gdf_plot = gdf_plot.dropna(subset=[value_column])

vmin, vmax = 0, 50000
tick_values = list(range(0, 50001, 5000))

m_dam_ev4 = folium.Map(
    location=[gdf_plot.geometry.centroid.y.mean(), gdf_plot.geometry.centroid.x.mean()],
    zoom_start=12,
    tiles="OpenStreetMap",
    width=900,
    height=600,
)

colormap = cm.LinearColormap(
    colors=cm.linear.YlOrRd_09.colors,
    vmin=vmin,
    vmax=vmax,
    caption="cost $",
).to_step(index=tick_values)

def style_function(feature):
    value = feature["properties"].get(value_column)
    if value is None or pd.isna(value):
        color = "#bdbdbd"
    else:
        value = max(vmin, min(vmax, float(value)))
        color = colormap(value)
    return {"color": color, "weight": 3, "opacity": 0.95}

folium.GeoJson(
    data=gdf_plot.to_json(),
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=[value_column], aliases=["cost $:"]),
).add_to(m_dam_ev4)

colormap.add_to(m_dam_ev4)
m_dam_ev4